<a href="https://colab.research.google.com/github/firuztahsinrodshi/traveliq-ota-business-intelligence/blob/main/notebooks_01_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# TravelIQ OTA — ETL Pipeline
# Notebook 1: Data Loading, Cleaning, Database Creation
# Author: Firuz Tahsin Rodshi

import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
from datetime import datetime, timedelta
import random

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

print("=" * 50)
print("TravelIQ OTA — ETL Pipeline")
print("=" * 50)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

TravelIQ OTA — ETL Pipeline
Started: 2026-07-02 19:10


In [2]:
from google.colab import files

print("Step 1: Upload hotel_bookings.csv")
uploaded = files.upload()

print("\nStep 2: Upload Hotel_Reviews.csv")
uploaded2 = files.upload()

Step 1: Upload hotel_bookings.csv


Saving hotel_bookings.csv to hotel_bookings.csv

Step 2: Upload Hotel_Reviews.csv


Saving Hotel_Reviews.csv to Hotel_Reviews.csv


In [3]:
print("Loading datasets...")
bookings = pd.read_csv('hotel_bookings.csv')
reviews = pd.read_csv('Hotel_Reviews.csv')

print(f"\n{'='*50}")
print(f"BOOKINGS DATASET")
print(f"{'='*50}")
print(f"Shape: {bookings.shape[0]:,} rows × {bookings.shape[1]} columns")
print(f"Memory: {bookings.memory_usage().sum() / 1024**2:.1f} MB")
print(f"\nNull values:")
nulls = bookings.isnull().sum()
print(nulls[nulls > 0])

print(f"\n{'='*50}")
print(f"REVIEWS DATASET")
print(f"{'='*50}")
print(f"Shape: {reviews.shape[0]:,} rows × {reviews.shape[1]} columns")
print(f"Columns: {reviews.columns.tolist()}")

Loading datasets...

BOOKINGS DATASET
Shape: 119,390 rows × 32 columns
Memory: 29.1 MB

Null values:
children         4
country        488
agent        16340
company     112593
dtype: int64

REVIEWS DATASET
Shape: 515,738 rows × 17 columns
Columns: ['Hotel_Address', 'Additional_Number_of_Scoring', 'Review_Date', 'Average_Score', 'Hotel_Name', 'Reviewer_Nationality', 'Negative_Review', 'Review_Total_Negative_Word_Counts', 'Total_Number_of_Reviews', 'Positive_Review', 'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags', 'days_since_review', 'lat', 'lng']


In [4]:
print("Cleaning bookings dataset...")
df = bookings.copy()

# Fix nulls
df['children'] = df['children'].fillna(0).astype(int)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

# Remove impossible records
before = len(df)
df = df[(df['adults'] + df['children'] + df['babies']) > 0]
df = df[df['adr'] >= 0]
df = df[~((df['adr'] == 0) & (df['is_canceled'] == 0))]
after = len(df)
print(f"Removed {before - after:,} invalid rows ({(before-after)/before*100:.1f}%)")

# Build arrival date
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d', errors='coerce'
)

# Derived columns
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_revenue'] = np.where(
    df['is_canceled'] == 0,
    df['adr'] * df['total_nights'], 0
)
df['booking_date'] = pd.to_datetime(
    df['reservation_status_date'], errors='coerce'
)

print(f"\nCleaned dataset: {len(df):,} rows")
print(f"Date range: {df['arrival_date'].min().date()} → {df['arrival_date'].max().date()}")
print(f"Cancellation rate: {df['is_canceled'].mean()*100:.1f}%")
print(f"Avg ADR: ${df[df['is_canceled']==0]['adr'].mean():.2f}")
print(f"Total revenue: ${df['total_revenue'].sum():,.0f}")

Cleaning bookings dataset...
Removed 1,803 invalid rows (1.5%)

Cleaned dataset: 117,587 rows
Date range: 2015-07-01 → 2017-08-31
Cancellation rate: 37.6%
Avg ADR: $102.38
Total revenue: $25,986,976


In [5]:
print("Cleaning reviews dataset...")
rev = reviews.copy()

# Keep relevant columns
keep_cols = ['Hotel_Name', 'Reviewer_Nationality', 'Reviewer_Score',
             'Positive_Review', 'Negative_Review',
             'Total_Number_of_Reviews_Reviewer_Has_Given',
             'Review_Date']
rev = rev[[c for c in keep_cols if c in rev.columns]]

# Clean text
rev['Positive_Review'] = rev['Positive_Review'].str.strip()
rev['Negative_Review'] = rev['Negative_Review'].str.strip()
rev['Positive_Review'] = rev['Positive_Review'].replace('No Positive', '')
rev['Negative_Review'] = rev['Negative_Review'].replace('No Negative', '')

# Sentiment label
rev['sentiment_label'] = pd.cut(
    rev['Reviewer_Score'],
    bins=[0, 5, 7, 10],
    labels=['Negative', 'Neutral', 'Positive']
).astype(str)

# Word count
rev['review_word_count'] = (
    rev['Positive_Review'].fillna('').str.split().str.len() +
    rev['Negative_Review'].fillna('').str.split().str.len()
)

print(f"Reviews cleaned: {len(rev):,} rows")
print(f"\nSentiment distribution:")
print(rev['sentiment_label'].value_counts())
print(f"\nAvg reviewer score: {rev['Reviewer_Score'].mean():.2f}/10")

Cleaning reviews dataset...
Reviews cleaned: 515,738 rows

Sentiment distribution:
sentiment_label
Positive    428476
Neutral      56559
Negative     30703
Name: count, dtype: int64

Avg reviewer score: 8.40/10


In [6]:
print("Building date dimension table...")

date_range = pd.date_range(start='2015-01-01', end='2017-12-31', freq='D')

seasons = {12: 'Winter', 1: 'Winter', 2: 'Winter',
           3: 'Spring', 4: 'Spring', 5: 'Spring',
           6: 'Summer', 7: 'Summer', 8: 'Summer',
           9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}

dim_date = pd.DataFrame({
    'date_id': range(len(date_range)),
    'full_date': date_range.strftime('%Y-%m-%d'),
    'year': date_range.year,
    'quarter': date_range.quarter,
    'month': date_range.month,
    'month_name': date_range.strftime('%B'),
    'week_of_year': date_range.isocalendar().week.values,
    'day_of_month': date_range.day,
    'day_of_week': date_range.dayofweek,
    'day_name': date_range.strftime('%A'),
    'is_weekend': (date_range.dayofweek >= 5).astype(int),
    'season': [seasons[m] for m in date_range.month]
})

print(f"Date dimension: {len(dim_date):,} rows")
print(f"Range: {dim_date['full_date'].iloc[0]} → {dim_date['full_date'].iloc[-1]}")



Building date dimension table...
Date dimension: 1,096 rows
Range: 2015-01-01 → 2017-12-31


In [7]:
print("Generating simulated supporting data...")

# Marketing campaigns
channels = ['Google Ads', 'Email', 'Social Media', 'Affiliate', 'Direct SEO']
segments = ['Leisure', 'Business', 'Groups', 'Couples', 'Families']
campaigns_list = []

for month_offset in range(30):
    date = pd.Timestamp('2015-07-01') + pd.DateOffset(months=month_offset)
    for channel in channels:
        budget = random.randint(8000, 60000)
        impressions = random.randint(50000, 800000)
        clicks = int(impressions * random.uniform(0.025, 0.09))
        bookings = int(clicks * random.uniform(0.04, 0.14))
        revenue = bookings * random.uniform(180, 520)
        cac = budget / max(bookings, 1)
        roas = revenue / max(budget, 1)
        campaigns_list.append({
            'campaign_name': f'{channel} — {date.strftime("%b %Y")}',
            'channel': channel,
            'start_date': str(date.date()),
            'end_date': str((date + pd.DateOffset(months=1)).date()),
            'target_segment': random.choice(segments),
            'budget_usd': budget,
            'impressions': impressions,
            'clicks': clicks,
            'bookings_attributed': bookings,
            'revenue_attributed': round(revenue, 2),
            'cac': round(cac, 2),
            'roas': round(roas, 2)
        })

fact_marketing = pd.DataFrame(campaigns_list)
print(f"Marketing campaigns: {len(fact_marketing):,} rows")
print(f"Total marketing spend: ${fact_marketing['budget_usd'].sum():,.0f}")
print(f"Avg ROAS: {fact_marketing['roas'].mean():.2f}x")

# Booking funnel
devices = ['Mobile', 'Desktop', 'Tablet']
funnel_list = []

for month_offset in range(30):
    date = pd.Timestamp('2015-07-01') + pd.DateOffset(months=month_offset)
    for segment in ['Direct', 'OTA', 'Corporate', 'Groups', 'Leisure']:
        for device in devices:
            # Mobile converts worse (realistic)
            conv_rate = {'Mobile': 0.025, 'Desktop': 0.065, 'Tablet': 0.045}[device]
            searches = random.randint(8000, 80000)
            views = int(searches * random.uniform(0.35, 0.65))
            wishlists = int(views * random.uniform(0.15, 0.35))
            started = int(wishlists * random.uniform(0.25, 0.55))
            payment = int(started * random.uniform(0.45, 0.75))
            completed = int(payment * conv_rate * 15)
            completed = min(completed, payment)
            overall_cvr = round(completed / max(searches, 1) * 100, 3)
            funnel_list.append({
                'session_date': str(date.date()),
                'device_id': devices.index(device) + 1,
                'market_segment': segment,
                'country_id': random.randint(1, 20),
                'searches': searches,
                'hotel_views': views,
                'wishlists': wishlists,
                'booking_started': started,
                'payment_reached': payment,
                'booking_completed': completed,
                'search_to_view_rate': round(views/max(searches,1)*100, 2),
                'view_to_book_rate': round(completed/max(views,1)*100, 2),
                'overall_conversion_rate': overall_cvr
            })

fact_funnel = pd.DataFrame(funnel_list)
print(f"\nFunnel sessions: {len(fact_funnel):,} rows")
print(f"Avg overall CVR: {fact_funnel['overall_conversion_rate'].mean():.2f}%")

Generating simulated supporting data...
Marketing campaigns: 150 rows
Total marketing spend: $5,238,123
Avg ROAS: 26.11x

Funnel sessions: 450 rows
Avg overall CVR: 2.07%


In [10]:
print("Loading all data into SQLite database...")

conn = sqlite3.connect('traveliq.db')

# Schema defined directly (mirrors schema.sql on GitHub)
schema_sql = """
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER, quarter INTEGER, month INTEGER,
    month_name TEXT, week_of_year INTEGER,
    day_of_month INTEGER, day_of_week INTEGER,
    day_name TEXT, is_weekend INTEGER,
    is_holiday INTEGER DEFAULT 0, season TEXT
);

CREATE TABLE IF NOT EXISTS dim_country (
    country_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_code TEXT NOT NULL UNIQUE,
    country_name TEXT, region TEXT,
    subregion TEXT, is_eu INTEGER DEFAULT 0
);

CREATE TABLE IF NOT EXISTS dim_hotel (
    hotel_id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_name TEXT, hotel_type TEXT,
    country_id INTEGER, star_rating REAL, total_rooms INTEGER,
    FOREIGN KEY (country_id) REFERENCES dim_country(country_id)
);

CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_id INTEGER, customer_type TEXT,
    is_repeated_guest INTEGER DEFAULT 0,
    total_previous_bookings INTEGER DEFAULT 0,
    total_previous_cancellations INTEGER DEFAULT 0,
    customer_segment TEXT,
    FOREIGN KEY (country_id) REFERENCES dim_country(country_id)
);

CREATE TABLE IF NOT EXISTS dim_room (
    room_id INTEGER PRIMARY KEY AUTOINCREMENT,
    room_type TEXT NOT NULL, room_category TEXT,
    max_occupancy INTEGER, base_rate REAL
);

CREATE TABLE IF NOT EXISTS dim_device (
    device_id INTEGER PRIMARY KEY AUTOINCREMENT,
    device_type TEXT, os_type TEXT, browser TEXT
);

CREATE TABLE IF NOT EXISTS fact_bookings (
    booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_id INTEGER, customer_id INTEGER, room_id INTEGER,
    arrival_date_id INTEGER, booking_date_id INTEGER,
    lead_time INTEGER, nights_weekend INTEGER DEFAULT 0,
    nights_weekday INTEGER DEFAULT 0, total_nights INTEGER,
    adults INTEGER DEFAULT 1, children INTEGER DEFAULT 0,
    babies INTEGER DEFAULT 0, total_guests INTEGER,
    meal_plan TEXT, market_segment TEXT, distribution_channel TEXT,
    is_cancelled INTEGER DEFAULT 0, adr REAL, total_revenue REAL,
    required_parking INTEGER DEFAULT 0, special_requests INTEGER DEFAULT 0,
    reservation_status TEXT, deposit_type TEXT, agent_id INTEGER,
    FOREIGN KEY (hotel_id) REFERENCES dim_hotel(hotel_id),
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (arrival_date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY (room_id) REFERENCES dim_room(room_id)
);

CREATE TABLE IF NOT EXISTS fact_reviews (
    review_id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_id INTEGER, reviewer_nationality TEXT,
    reviewer_score REAL, positive_review TEXT,
    negative_review TEXT, review_word_count INTEGER,
    total_reviews_by_reviewer INTEGER, review_date TEXT,
    sentiment_label TEXT, sentiment_score REAL,
    FOREIGN KEY (hotel_id) REFERENCES dim_hotel(hotel_id)
);

CREATE TABLE IF NOT EXISTS fact_marketing (
    campaign_id INTEGER PRIMARY KEY AUTOINCREMENT,
    campaign_name TEXT, channel TEXT, start_date TEXT,
    end_date TEXT, target_segment TEXT, budget_usd REAL,
    impressions INTEGER, clicks INTEGER,
    bookings_attributed INTEGER, revenue_attributed REAL,
    cac REAL, roas REAL
);

CREATE TABLE IF NOT EXISTS fact_funnel (
    funnel_id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_date TEXT, device_id INTEGER, market_segment TEXT,
    country_id INTEGER, searches INTEGER, hotel_views INTEGER,
    wishlists INTEGER, booking_started INTEGER,
    payment_reached INTEGER, booking_completed INTEGER,
    search_to_view_rate REAL, view_to_book_rate REAL,
    overall_conversion_rate REAL,
    FOREIGN KEY (device_id) REFERENCES dim_device(device_id)
);

CREATE TABLE IF NOT EXISTS fact_payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    booking_id INTEGER, payment_method TEXT,
    payment_status TEXT, amount_usd REAL,
    currency TEXT DEFAULT 'USD', payment_date TEXT,
    refund_amount REAL DEFAULT 0, is_refunded INTEGER DEFAULT 0,
    FOREIGN KEY (booking_id) REFERENCES fact_bookings(booking_id)
);
"""

# Execute each statement
for stmt in schema_sql.split(';'):
    s = stmt.strip()
    if s and not s.startswith('--'):
        try:
            conn.execute(s)
        except Exception as e:
            pass
conn.commit()
print("✅ Schema created successfully")

# ── Load fact_bookings ──────────────────────────────────────────
df_load = df.copy()
df_load['hotel_id'] = (df_load['hotel'] == 'City Hotel').astype(int) + 1
df_load['customer_id'] = range(1, len(df_load) + 1)
df_load['room_id'] = df_load['reserved_room_type'].apply(
    lambda x: ord(x) - ord('A') + 1 if isinstance(x, str) else 1
)
df_load['arrival_date_id'] = df_load['arrival_date'].apply(
    lambda x: int((x - pd.Timestamp('2015-01-01')).days)
    if pd.notnull(x) else 0
)
df_load['arrival_date'] = df_load['arrival_date'].astype(str)
df_load['booking_date'] = df_load['booking_date'].astype(str)

df_load = df_load.rename(columns={
    'is_canceled':                   'is_cancelled',
    'stays_in_weekend_nights':       'nights_weekend',
    'stays_in_week_nights':          'nights_weekday',
    'required_car_parking_spaces':   'required_parking',
    'total_of_special_requests':     'special_requests',
    'meal':                          'meal_plan',
})

load_cols = [
    'hotel_id', 'customer_id', 'room_id', 'arrival_date_id',
    'lead_time', 'nights_weekend', 'nights_weekday', 'total_nights',
    'adults', 'children', 'babies', 'total_guests',
    'meal_plan', 'market_segment', 'distribution_channel',
    'is_cancelled', 'adr', 'total_revenue',
    'required_parking', 'special_requests',
    'reservation_status', 'deposit_type'
]

df_load[[c for c in load_cols if c in df_load.columns]].to_sql(
    'fact_bookings', conn, if_exists='replace', index=False
)
print(f"✅ fact_bookings: {len(df_load):,} rows loaded")

# ── Load fact_reviews ───────────────────────────────────────────
rev_load = rev.rename(columns={
    'Hotel_Name':                                   'hotel_name',
    'Reviewer_Nationality':                         'reviewer_nationality',
    'Reviewer_Score':                               'reviewer_score',
    'Positive_Review':                              'positive_review',
    'Negative_Review':                              'negative_review',
    'Total_Number_of_Reviews_Reviewer_Has_Given':   'total_reviews_by_reviewer',
    'Review_Date':                                  'review_date',
})
rev_load['hotel_id'] = 1
rev_load['sentiment_score'] = (rev_load['reviewer_score'] - 5) / 5
rev_load.to_sql('fact_reviews', conn, if_exists='replace', index=False)
print(f"✅ fact_reviews: {len(rev_load):,} rows loaded")

# ── Load dim_date ───────────────────────────────────────────────
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
print(f"✅ dim_date: {len(dim_date):,} rows loaded")

# ── Load fact_marketing ─────────────────────────────────────────
fact_marketing.to_sql('fact_marketing', conn, if_exists='replace', index=False)
print(f"✅ fact_marketing: {len(fact_marketing):,} rows loaded")

# ── Load fact_funnel ────────────────────────────────────────────
fact_funnel.to_sql('fact_funnel', conn, if_exists='replace', index=False)
print(f"✅ fact_funnel: {len(fact_funnel):,} rows loaded")

conn.commit()

# ── Verification ────────────────────────────────────────────────
print("\n" + "=" * 50)
print("DATABASE VERIFICATION")
print("=" * 50)
for table in ['fact_bookings', 'fact_reviews', 'fact_marketing',
              'fact_funnel', 'dim_date']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<25} {count:>10,} rows")

conn.close()
print("\n✅ Database complete — traveliq.db is ready")
print("📥 Download it from the Files panel (left sidebar folder icon)")

Loading all data into SQLite database...
✅ Schema created successfully
✅ fact_bookings: 117,587 rows loaded
✅ fact_reviews: 515,738 rows loaded
✅ dim_date: 1,096 rows loaded
✅ fact_marketing: 150 rows loaded
✅ fact_funnel: 450 rows loaded

DATABASE VERIFICATION
  fact_bookings                117,587 rows
  fact_reviews                 515,738 rows
  fact_marketing                   150 rows
  fact_funnel                      450 rows
  dim_date                       1,096 rows

✅ Database complete — traveliq.db is ready
📥 Download it from the Files panel (left sidebar folder icon)
